# Fine-Tune a Model Router using Azure OpenAI REST API

This notebook walks you through end-to-end fine-tuning of a **Model Router** using Azure OpenAI REST endpoints. The workflow follows the official [Azure OpenAI fine-tuning guide](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/fine-tuning?tabs=oai-sdk&pivots=rest-api#review-the-workflow-for-the-rest-api).

## Workflow
1. **Configure** – Set all parameters in a single config cell.
2. **Understand the data format** – Review how training data is structured.
3. **Upload data** – Upload training (and optionally test) files to Azure OpenAI.
4. **Submit fine-tuning job** – Kick off model training.
5. **Monitor job** – Poll until training completes.
6. **Download results** – Retrieve the training metrics CSV.
7. **Deploy the model** – Create a deployment for inference.
8. **Test the deployment** – Make a test call to verify the fine-tuned model.

## Prerequisites
- An Azure OpenAI resource with fine-tuning access.
- The **Azure AI Owner** role (required for deployment).
- Python packages: `requests`.
- An API key or Azure AD token for your resource.

## 0. Imports

All imports are consolidated here at the top of the notebook.

In [ ]:
import json
import os
import sys

# Ensure the src/ helper module is importable
sys.path.insert(0, os.path.abspath("."))

from src.finetuning_helpers import (
    upload_file,
    create_finetuning_job,
    get_job_status,
    poll_job_until_complete,
    list_job_events,
    download_result_file,
    deploy_finetuned_model,
)

## 1. Configuration

Set all user-configurable parameters below. This is the **only cell** you need to edit before running the notebook.

In [ ]:
# ── Azure OpenAI Resource ──────────────────────────────────────────────
AZURE_OPENAI_ENDPOINT = "https://<YOUR_RESOURCE_NAME>.openai.azure.com"  # e.g. https://myresource.openai.azure.com
AZURE_OPENAI_API_KEY  = "<YOUR_API_KEY>"  # Or set via environment variable

# ── Data Paths ─────────────────────────────────────────────────────────
TRAINING_FILE_PATH   = "data/bigbench_hard_train_sample200.jsonl"
VALIDATION_FILE_PATH = "data/bigbench_hard_test.jsonl"  # Optional – if not provided, training data is split 80:20 automatically

# ── Model ──────────────────────────────────────────────────────────────
BASE_MODEL = "gpt-4.1-mini-2025-04-14"  # Base model to fine-tune
SUFFIX     = "model-router"             # Up to 18 chars; helps identify your fine-tuned model

# ── Hyperparameters (Optional – leave as None to use defaults) ────────
N_EPOCHS                 = None  # e.g. 3
BATCH_SIZE               = None  # e.g. 4
LEARNING_RATE_MULTIPLIER = None  # e.g. 0.1
SEED                     = 105   # For reproducibility

# ── Deployment (used after training succeeds) ─────────────────────────
SUBSCRIPTION_ID      = "<YOUR_SUBSCRIPTION_ID>"
RESOURCE_GROUP       = "<YOUR_RESOURCE_GROUP>"
RESOURCE_NAME        = "<YOUR_RESOURCE_NAME>"  # Same as in the endpoint
DEPLOYMENT_NAME      = "model-router-ft"       # Name for the deployed model
DEPLOYMENT_SKU       = "standard"               # "standard" or "globalbatch"
DEPLOYMENT_CAPACITY  = 1

# ── Monitoring ─────────────────────────────────────────────────────────
POLL_INTERVAL_SECONDS = 60  # How often to check job status

## 2. Understanding the Data Format

The training data must be in **JSONL** (JSON Lines) format. Each line is a JSON object with the following structure:

```json
{
  "messages": [
    {"role": "user", "content": "<your prompt>"}
  ],
  "metrics": {
    "model_a": 1,
    "model_b": 0
  },
  "usage": {
    "model_a": {"completion_tokens": 205, "prompt_tokens": 184},
    "model_b": {"completion_tokens": 179, "prompt_tokens": 184}
  }
}
```

### Field descriptions

| Field | Required | Description |
|-------|----------|-------------|
| `messages` | ✅ Yes | A list of chat messages following the Chat Completions API format. Typically contains a `user` message with the prompt. |
| `metrics` | ✅ Yes | A dictionary mapping each model name to a binary score (`1` = correct, `0` = incorrect). The model router uses these to learn which models perform well on which types of tasks. |
| `usage` | ❌ Optional | A dictionary mapping each model name to its token usage (`completion_tokens` and `prompt_tokens`). When provided, it helps the router optimize for cost-efficiency alongside accuracy. |

### Notes
- **Validation data is optional.** If you do not provide a validation file, the training data is automatically split **80:20** (80% train, 20% validation).
- Files must be UTF-8 encoded and less than 512 MB.
- A minimum of 10 training examples is required, but **hundreds or more** are recommended for meaningful results.

Let's preview a sample from the training data:

In [ ]:
# Preview the first training example
with open(TRAINING_FILE_PATH, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print("=== Sample Training Record ===")
print(json.dumps(sample, indent=2)[:2000])  # Truncate for readability

## 3. Upload Data

Upload the training file (and optionally the validation file) to Azure OpenAI.

In [ ]:
# Upload training file
print("Uploading training file...")
train_response = upload_file(AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, TRAINING_FILE_PATH)
training_file_id = train_response["id"]
print(f"Training file ID: {training_file_id}")

# Upload validation file (optional)
validation_file_id = None
if VALIDATION_FILE_PATH:
    print("Uploading validation file...")
    val_response = upload_file(AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, VALIDATION_FILE_PATH)
    validation_file_id = val_response["id"]
    print(f"Validation file ID: {validation_file_id}")
else:
    print("No validation file provided. Training data will be split 80:20 automatically.")

## 4. Submit Fine-Tuning Job

Create a fine-tuning job with the uploaded data. Hyperparameters are optional — the service uses sensible defaults if not specified.

In [ ]:
print("Submitting fine-tuning job...")
job_response = create_finetuning_job(
    endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    model=BASE_MODEL,
    training_file_id=training_file_id,
    validation_file_id=validation_file_id,
    suffix=SUFFIX,
    seed=SEED,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate_multiplier=LEARNING_RATE_MULTIPLIER,
)

job_id = job_response["id"]
print(f"Fine-tuning job submitted!")
print(f"Job ID: {job_id}")
print(json.dumps(job_response, indent=2))

## 5. Monitor the Fine-Tuning Job

Poll the job status until it reaches a terminal state (`succeeded`, `failed`, or `cancelled`). This can take minutes to hours depending on dataset size and model.

In [ ]:
# Poll until the job finishes
final_status = poll_job_until_complete(
    endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    job_id=job_id,
    poll_interval=POLL_INTERVAL_SECONDS,
)

print("\n=== Final Job Status ===")
print(json.dumps(final_status, indent=2))

In [ ]:
# View training events / logs
events = list_job_events(AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, job_id)
print(json.dumps(events, indent=2))

## 6. Download Result File

After a successful job, Azure OpenAI generates a `results.csv` with training metrics (loss, accuracy per step/epoch). Let's download it.

In [ ]:
if final_status["status"] == "succeeded":
    result_file_id = final_status["result_files"][0]
    print(f"Result file ID: {result_file_id}")

    download_result_file(
        endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        file_id=result_file_id,
        output_path="results.csv",
    )

    # Preview the results
    import pandas as pd
    df = pd.read_csv("results.csv")
    print(df.head(10))
else:
    print(f"Job did not succeed. Status: {final_status['status']}")
    print("Check the events above for error details.")

## 7. Deploy the Fine-Tuned Model

Deployment uses the **Azure Management REST API** (control plane), which requires an Azure AD bearer token rather than an API key.

Generate a token by running in your terminal:
```bash
az login
az account get-access-token --query accessToken -o tsv
```

> **Note:** You need the **Azure AI Owner** role to create deployments.

> ⚠️ **Important:** The **subset** feature and **mode** feature are **not supported** for fine-tuned model deployments. All inference requests will only be routed to the models present in the training data.

In [ ]:
# Get the fine-tuned model name from the completed job
fine_tuned_model = final_status.get("fine_tuned_model")
print(f"Fine-tuned model: {fine_tuned_model}")

# Set your Azure AD token here
AZURE_AD_TOKEN = "<PASTE_YOUR_TOKEN_HERE>"  # from: az account get-access-token --query accessToken -o tsv

if fine_tuned_model:
    print(f"Deploying model '{fine_tuned_model}' as '{DEPLOYMENT_NAME}'...")
    deploy_response = deploy_finetuned_model(
        token=AZURE_AD_TOKEN,
        subscription_id=SUBSCRIPTION_ID,
        resource_group=RESOURCE_GROUP,
        resource_name=RESOURCE_NAME,
        deployment_name=DEPLOYMENT_NAME,
        fine_tuned_model=fine_tuned_model,
        sku_name=DEPLOYMENT_SKU,
        sku_capacity=DEPLOYMENT_CAPACITY,
    )
    print("Deployment created successfully!")
    print(json.dumps(deploy_response, indent=2))
else:
    print("No fine-tuned model available. Ensure the training job succeeded.")

## 8. Test the Fine-Tuned Deployment

Make a single test call to the deployed fine-tuned model using the Chat Completions REST API to verify it is working correctly.

In [ ]:
# Read the first example from the test file (or training file) to use as a test prompt
test_file = VALIDATION_FILE_PATH or TRAINING_FILE_PATH
with open(test_file, "r", encoding="utf-8") as f:
    test_sample = json.loads(f.readline())

test_messages = test_sample["messages"]
print("=== Test Prompt ===")
print(test_messages[0]["content"][:500])

# Call the deployed fine-tuned model
url = f"{AZURE_OPENAI_ENDPOINT}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions?api-version=2025-04-01-preview"
headers = {"Content-Type": "application/json", "api-key": AZURE_OPENAI_API_KEY}
payload = {"messages": test_messages}

response = requests.post(url, headers=headers, json=payload)
response.raise_for_status()
result = response.json()

print("\n=== Model Response ===")
print(result["choices"][0]["message"]["content"])
print(f"\nTokens used — prompt: {result['usage']['prompt_tokens']}, completion: {result['usage']['completion_tokens']}")

## Next Steps

- **Use the model** via the Chat Completions API with your `DEPLOYMENT_NAME`.
- **Continuous fine-tuning**: Use the fine-tuned model ID as the base model for further iterations.
- **Clean up**: Delete unused deployments to avoid ongoing hosting costs (deployments inactive for 15+ days are auto-deleted).